# Devoir 1 — NLP Pipeline (Arabic Text)

Ce notebook applique une pipeline NLP simple sur le fichier `arabic_nlp_text.txt`.

## Objectifs
- Charger et inspecter le texte.
- Nettoyer/normaliser du texte arabe.
- Tokeniser en mots.
- Filtrer les stopwords.
- Extraire des statistiques et mots fréquents.
- Segmenter en phrases.

Exécute les cellules **dans l’ordre**.

In [1]:
# Imports for the Arabic NLP pipeline.
# All third-party packages are listed in ../requirements.txt.
from pathlib import Path
import re
from collections import Counter

import nltk
from nltk.corpus import stopwords

TEXT_PATH = Path('arabic_nlp_text.txt')
print(f'Notebook ready. Target file: {TEXT_PATH.resolve()}')


Notebook ready. Target file: /Users/beethoven/NLP/devoir0/arabic_nlp_text.txt


## 1) Chargement du texte

Dans cette section, on lit le fichier source en UTF-8 et on affiche un aperçu.

**Pourquoi ?** Vérifier que le texte est bien lu avant de lancer le traitement.

In [2]:
assert TEXT_PATH.exists(), f"Fichier introuvable: {TEXT_PATH}"

raw_text = TEXT_PATH.read_text(encoding="utf-8")

print("Longueur (caractères):", len(raw_text))
print("Longueur (mots bruts):", len(raw_text.split()))
print("\nAperçu:\n")
print(raw_text[:700])

Longueur (caractères): 27224
Longueur (mots bruts): 4635

Aperçu:

في مدينةٍ قديمةٍ تحيط بها الأسوار العالية، كان الناس يعيشون حياةً بسيطة
ولكنها مليئة بالحكايات. كان السوق المركزي هو قلب المدينة النابض، حيث
تختلط الأصوات والروائح والألوان في مشهدٍ لا يُنسى. التجار ينادون على
بضائعهم، والأطفال يركضون بين الأزقة، والنساء يتبادلن الأخبار اليومية.

في أحد الأيام، وصل غريب إلى المدينة. كان يرتدي عباءة طويلة ويحمل حقيبة
صغيرة. لم يكن أحد يعرف من أين جاء، لكنه سرعان ما أصبح موضوع حديث الجميع.
جلس في المقهى القديم، يراقب الناس بصمت، وكأنه يبحث عن شيءٍ ما.

مرّت الأيام، وبدأ الغريب يتحدث مع بعض السكان. اكتشفوا أنه رجل متعلم، يعرف
الكثير عن اللغات والتاريخ. كان يحكي قصصًا عن مدن بعيدة، وعن حضارات
اندثرت، وعن رحلاتٍ عبر الصحارى والبحار. أصبح الناس يتجمعون حوله كل مسا


## 2) Normalisation du texte arabe

Ici, on simplifie le texte pour l’analyse :
- suppression des diacritiques,
- unification de certaines lettres arabes,
- nettoyage de la ponctuation non utile.

**Pourquoi ?** Réduire les variations d’écriture qui faussent les fréquences.

In [3]:
def normalize_arabic(text: str) -> str:
    text = text.strip()

    # Supprimer les diacritiques arabes (tashkeel)
    text = re.sub(r"[\u0617-\u061A\u064B-\u0652]", "", text)

    # Normalisation de lettres
    text = re.sub(r"[إأآا]", "ا", text)
    text = re.sub(r"ى", "ي", text)
    text = re.sub(r"ؤ", "و", text)
    text = re.sub(r"ئ", "ي", text)
    text = re.sub(r"ة", "ه", text)

    # Supprimer tatweel
    text = re.sub(r"ـ", "", text)

    # Garder lettres arabes, chiffres, espaces et ponctuation de base
    text = re.sub(r"[^\u0600-\u06FF0-9\s\.!؟،]", " ", text)

    # Normaliser les espaces
    text = re.sub(r"\s+", " ", text).strip()
    return text

clean_text = normalize_arabic(raw_text)

print("Aperçu normalisé:\n")
print(clean_text[:700])

Aperçu normalisé:

في مدينه قديمه تحيط بها الاسوار العاليه، كان الناس يعيشون حياه بسيطه ولكنها ملييه بالحكايات. كان السوق المركزي هو قلب المدينه النابض، حيث تختلط الاصوات والروايح والالوان في مشهد لا ينسي. التجار ينادون علي بضايعهم، والاطفال يركضون بين الازقه، والنساء يتبادلن الاخبار اليوميه. في احد الايام، وصل غريب الي المدينه. كان يرتدي عباءه طويله ويحمل حقيبه صغيره. لم يكن احد يعرف من اين جاء، لكنه سرعان ما اصبح موضوع حديث الجميع. جلس في المقهي القديم، يراقب الناس بصمت، وكانه يبحث عن شيء ما. مرت الايام، وبدا الغريب يتحدث مع بعض السكان. اكتشفوا انه رجل متعلم، يعرف الكثير عن اللغات والتاريخ. كان يحكي قصصا عن مدن بعيده، وعن حضارات اندثرت، وعن رحلات عبر الصحاري والبحار. اصبح الناس يتجمعون حوله كل مساء ليستمعوا 


## 3) Tokenisation + Stopwords

On découpe le texte en mots arabes puis on retire les mots très fréquents (stopwords) comme "في" ou "من".

**Pourquoi ?** Mieux faire ressortir les mots informatifs du corpus.

In [4]:
# Try to load NLTK Arabic stopwords; fall back to a small inline set so the
# notebook still runs offline (e.g. in a sandboxed grading environment).
try:
    nltk.download('stopwords', quiet=True)
    arabic_stopwords = set(stopwords.words('arabic'))
    sw_source = 'NLTK'
except (LookupError, OSError):
    arabic_stopwords = set()
    sw_source = 'NLTK download failed'

if not arabic_stopwords:
    ARABIC_STOPWORDS_FALLBACK = {
        'في','من','إلى','على','عن','ما','لا','إن','أن','إذ','كل','بعض','مع','أو','ثم','قد','هذا','هذه','ذلك','تلك',
        'كان','كانت','يكون','أين','كيف','متى','هل','نعم','بعد','قبل','حيث','عند','بين','أمام','خلف','فوق','تحت',
        'يا','هو','هي','هم','نحن','أنا','أنت','الذي','التي','اللذان','اللتان','الذين','اللواتي','أي','أية',
        'لكن','بل','إذا','لو','حتى','منذ','ولا','وما','وقد','فقد','وكان','إنما','أيضا','حول','بسبب','رغم'
    }
    arabic_stopwords = ARABIC_STOPWORDS_FALLBACK
    sw_source = 'inline fallback'

print(f'Stopwords source: {sw_source} ({len(arabic_stopwords)} words)')

# Extract Arabic-only tokens from the normalized text.
all_tokens = re.findall(r'[\u0621-\u064A]+', clean_text)
filtered_tokens = [t for t in all_tokens if t not in arabic_stopwords and len(t) > 1]

print('Tokens before filter:', len(all_tokens))
print('Tokens after filter :', len(filtered_tokens))
print('\nSample filtered tokens:')
print(filtered_tokens[:40])


Stopwords source: NLTK (701 words)
Tokens before filter: 4635
Tokens after filter : 3750

Sample filtered tokens:
['مدينه', 'قديمه', 'تحيط', 'الاسوار', 'العاليه', 'الناس', 'يعيشون', 'حياه', 'بسيطه', 'ولكنها', 'ملييه', 'بالحكايات', 'السوق', 'المركزي', 'قلب', 'المدينه', 'النابض', 'تختلط', 'الاصوات', 'والروايح', 'والالوان', 'مشهد', 'ينسي', 'التجار', 'ينادون', 'علي', 'بضايعهم', 'والاطفال', 'يركضون', 'الازقه', 'والنساء', 'يتبادلن', 'الاخبار', 'اليوميه', 'احد', 'الايام', 'وصل', 'غريب', 'الي', 'المدينه']


## 4) Fréquences des mots (Top 20)

On calcule la fréquence des mots les plus présents.

**Pourquoi ?** Obtenir une vue rapide des thèmes dominants du texte.

In [5]:
freq = Counter(filtered_tokens)

print("Top 20 mots les plus fréquents:\n")
for word, count in freq.most_common(20):
    print(f"{word:<15} {count}")

Top 20 mots les plus fréquents:

المدينه         75
الي             75
الغريب          75
وعن             60
كانت            60
ليلي            60
الناس           45
احد             45
الايام          45
السوق           30
علي             30
طويله           30
يعرف            30
اصبح            30
المقهي          30
وكانه           30
مرت             30
الكثير          30
مدن             30
الوقت           30


## 5) Segmentation en phrases

Ici, on découpe le texte en phrases avec la ponctuation arabe/latine.

**Pourquoi ?** Préparer des analyses phrase par phrase (résumé, sentiment, extraction d’information, etc.).

In [6]:
sentences = [s.strip() for s in re.split(r"[\.!؟]+", clean_text) if s.strip()]

print("Nombre de phrases:", len(sentences))

avg_words = sum(len(re.findall(r"[\u0621-\u064A]+", s)) for s in sentences) / max(len(sentences), 1)
print("Moyenne de mots par phrase:", round(avg_words, 2))

print("\nExemples de phrases:")
for s in sentences[:5]:
    print("-", s[:120])

Nombre de phrases: 450
Moyenne de mots par phrase: 10.3

Exemples de phrases:
- في مدينه قديمه تحيط بها الاسوار العاليه، كان الناس يعيشون حياه بسيطه ولكنها ملييه بالحكايات
- كان السوق المركزي هو قلب المدينه النابض، حيث تختلط الاصوات والروايح والالوان في مشهد لا ينسي
- التجار ينادون علي بضايعهم، والاطفال يركضون بين الازقه، والنساء يتبادلن الاخبار اليوميه
- في احد الايام، وصل غريب الي المدينه
- كان يرتدي عباءه طويله ويحمل حقيبه صغيره


## 6) Mini extraction de mots-clés

On prend les mots les plus fréquents comme **mots-clés simples**.

**Pourquoi ?** Avoir une première idée des sujets importants du texte sans modèle complexe.

In [7]:
top_keywords = [w for w, _ in freq.most_common(10)]
print("Mots-clés (Top 10):")
print(top_keywords)

# Trouver une phrase exemple pour le premier mot-clé
if top_keywords:
    kw = top_keywords[0]
    example = next((s for s in sentences if kw in s), "Aucune phrase trouvée")
    print(f"\nExemple de phrase contenant '{kw}':")
    print(example[:250])

Mots-clés (Top 10):
['المدينه', 'الي', 'الغريب', 'وعن', 'كانت', 'ليلي', 'الناس', 'احد', 'الايام', 'السوق']

Exemple de phrase contenant 'المدينه':
كان السوق المركزي هو قلب المدينه النابض، حيث تختلط الاصوات والروايح والالوان في مشهد لا ينسي


## 7) Conclusion

Tu as maintenant une pipeline NLP de base pour l’arabe :
- lecture,
- normalisation,
- tokenisation,
- retrait des stopwords,
- statistiques,
- segmentation en phrases,
- mots-clés.

Ensuite, tu peux ajouter : n-grams, topic modeling, classification, résumé automatique.

## 8) Passage en mode professionnel : TF, IDF, TF-IDF (poids)

Ici, on passe d’une simple fréquence à une représentation vectorielle plus solide.

### Rappels théoriques
- $TF(t,d)$ : fréquence du terme $t$ dans le document $d$.
- $IDF(t)$ : importance inverse dans le corpus (plus un terme est rare, plus son IDF est élevé).
- $TF\text{-}IDF(t,d)=TF(t,d)\times IDF(t)$.

**Intérêt** : obtenir des **poids** pertinents par document et au niveau du corpus.

In [8]:
# Vectorization + similarity
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print('Vectorization libraries ready.')


Vectorization libraries ready.


## 9) Construire un corpus de documents

TF-IDF nécessite un **ensemble de documents** (pas un seul bloc).

Ici on crée des documents à partir des paragraphes, puis fallback sur des groupes de phrases si besoin.

In [9]:
# 1) Paragraphes comme documents
paragraphs = [p.strip() for p in raw_text.split("\n\n") if p.strip()]
documents = [normalize_arabic(p) for p in paragraphs if len(p.split()) >= 20]

# 2) Fallback: groupes de 3 phrases si trop peu de docs
if len(documents) < 5:
    sentence_docs = [s.strip() for s in re.split(r"[\.!؟]+", clean_text) if s.strip()]
    grouped = [" ".join(sentence_docs[i:i+3]) for i in range(0, len(sentence_docs), 3)]
    documents = [g for g in grouped if len(g.split()) >= 15]

print("Nombre de documents du corpus:", len(documents))
print("Extrait document[0]:\n")
print(documents[0][:250])

Nombre de documents du corpus: 135
Extrait document[0]:

في مدينه قديمه تحيط بها الاسوار العاليه، كان الناس يعيشون حياه بسيطه ولكنها ملييه بالحكايات. كان السوق المركزي هو قلب المدينه النابض، حيث تختلط الاصوات والروايح والالوان في مشهد لا ينسي. التجار ينادون علي بضايعهم، والاطفال يركضون بين الازقه، والنساء 


## 10) Matrice TF (Document-Term Matrix)

On calcule d’abord les **fréquences brutes** par document.

**Sortie attendue** : poids TF globaux pour repérer les termes dominants en volume.

In [10]:
# Stopwords normalisés pour cohérence avec le prétraitement
arabic_stopwords_sklearn = sorted({
    w_norm
    for w in arabic_stopwords
    for w_norm in [normalize_arabic(w)]
    if re.fullmatch(r"[\u0621-\u064A]{2,}", w_norm or "")
})

count_vectorizer = CountVectorizer(
    token_pattern=r"(?u)[\u0621-\u064A]{2,}",
    stop_words=arabic_stopwords_sklearn,
    ngram_range=(1, 2),  # unigrams + bigrams
    min_df=2
)

X_tf = count_vectorizer.fit_transform(documents)
terms_tf = count_vectorizer.get_feature_names_out()

tf_global = np.asarray(X_tf.sum(axis=0)).ravel()
tf_global_norm = tf_global / tf_global.sum()

tf_df = pd.DataFrame({
    "term": terms_tf,
    "tf_count": tf_global,
    "tf_weight": tf_global_norm
}).sort_values("tf_weight", ascending=False)

print("Taille matrice TF:", X_tf.shape)
tf_df.head(15)

Taille matrice TF: (135, 399)


,term,tf_count,tf_weight
54,الغريب,75,0.011547
82,المدينه,75,0.011547
240,كانت,60,0.009238
338,وعن,60,0.009238
263,ليلي,60,0.009238
26,الايام,45,0.006928
95,الناس,45,0.006928
383,يعرف,30,0.004619
237,كاتبه,30,0.004619
277,مرت,30,0.004619


## 11) Matrice TF-IDF et poids des termes

On calcule ensuite TF-IDF pour pondérer les termes selon leur caractère informatif.

- **idf élevé** : terme discriminant (plus rare).
- **tf-idf moyen élevé** : terme important dans l’ensemble du corpus.

In [11]:
tfidf_vectorizer = TfidfVectorizer(
    token_pattern=r"(?u)[\u0621-\u064A]{2,}",
    stop_words=arabic_stopwords_sklearn,
    ngram_range=(1, 2),
    min_df=2,
    norm="l2"
)

X_tfidf = tfidf_vectorizer.fit_transform(documents)
terms_tfidf = tfidf_vectorizer.get_feature_names_out()
idf_values = tfidf_vectorizer.idf_

tfidf_mean = np.asarray(X_tfidf.mean(axis=0)).ravel()

tfidf_df = pd.DataFrame({
    "term": terms_tfidf,
    "idf": idf_values,
    "mean_tfidf_weight": tfidf_mean
}).sort_values("mean_tfidf_weight", ascending=False)

print("Taille matrice TF-IDF:", X_tfidf.shape)
tfidf_df.head(20)

Taille matrice TF-IDF: (135, 399)


,term,idf,mean_tfidf_weight
338,وعن,2.478668,0.048132
240,كانت,2.084013,0.045370
82,المدينه,1.581922,0.043478
54,الغريب,1.581922,0.041727
263,ليلي,1.801781,0.041524
26,الايام,2.084013,0.032463
70,الكتاب,2.478668,0.030087
95,الناس,2.084013,0.029797
224,قررت,2.478668,0.028625
75,الكثير,2.478668,0.027602


## 12) Poids TF-IDF par document (lecture experte)

On inspecte les termes les plus importants pour un document donné.

**Pourquoi ?** C’est la base pour indexation, recherche d’information et classification.

In [12]:
doc_id = 0  # changer l'index pour analyser un autre document

row = X_tfidf[doc_id].toarray().ravel()
nonzero_idx = np.where(row > 0)[0]

doc_terms = pd.DataFrame({
    "term": terms_tfidf[nonzero_idx],
    "tfidf_weight": row[nonzero_idx]
}).sort_values("tfidf_weight", ascending=False)

print(f"Document analysé: {doc_id} | nb termes non nuls: {len(nonzero_idx)}")
doc_terms.head(20)

Document analysé: 0 | nb termes non nuls: 65


,term,tfidf_weight
0,الاخبار,0.125672
33,حياه,0.125672
35,قديمه,0.125672
36,قديمه تحيط,0.125672
37,قلب,0.125672
38,قلب المدينه,0.125672
39,مدينه,0.125672
40,مدينه قديمه,0.125672
41,مشهد,0.125672
42,مشهد ينسي,0.125672


## 13) Similarité cosinus entre documents

On mesure la proximité sémantique entre documents via les vecteurs TF-IDF.

$$\text{cosine}(a,b)=\frac{a\cdot b}{\|a\|\|b\|}$$

**Usage** : clustering, détection de doublons, moteur de recherche.

In [13]:
sim_matrix = cosine_similarity(X_tfidf)

print("Dimensions matrice similarité:", sim_matrix.shape)
print("\nSous-matrice (5x5):")
print(np.round(sim_matrix[:5, :5], 3))

# Top voisins du document 0
neighbors = np.argsort(-sim_matrix[0])
print("\nDocuments les plus proches de doc 0 (hors lui-même):", neighbors[1:6].tolist())

Dimensions matrice similarité: (135, 135)

Sous-matrice (5x5):
[[1.    0.013 0.007 0.011 0.   ]
 [0.013 1.    0.029 0.    0.013]
 [0.007 0.029 1.    0.005 0.052]
 [0.011 0.    0.005 1.    0.005]
 [0.    0.013 0.052 0.005 1.   ]]

Documents les plus proches de doc 0 (hors lui-même): [81, 45, 27, 99, 117]


## 14) Interprétation académique (à mettre dans le rapport)

- Les **poids TF** identifient les termes dominants en volume.
- Les **poids IDF** isolent les termes discriminants.
- Les **poids TF-IDF** équilibrent fréquence locale et rareté globale.
- La **similarité cosinus** permet d’évaluer la proximité thématique entre documents.

👉 Pour aller plus loin (niveau master) : LSA/SVD, topic modeling (LDA), embeddings (AraBERT/sentence-transformers arabes).